<a href="https://colab.research.google.com/github/barnaor97/reaserch/blob/main/01_data_to_gee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ============================================================
# Configuration
# ============================================================
# All asset paths and spatial parameters used across the
# notebook are defined here. The VERSION suffix is appended
# to every output asset name.
# ============================================================

VERSION = "_10m"

# Project
PROJECT_ID = "bar-project-457815"
ASSET_ROOT = f"projects/{PROJECT_ID}/assets"

# Spatial parameters
SCALE      = 10
CRS        = "EPSG:32636"
MAX_PIXELS = 1e13

# Cloud masking and temporal coverage
CLEAR_THRESH = 0.60
DATE_START   = "2017-01-01"
DATE_END     = "2025-12-31"

# Output asset paths
ASSET_COLLECTION       = f"{ASSET_ROOT}/s2_monthly_image_collection{VERSION}"
LABEL_COLLECTION       = f"{ASSET_ROOT}/fire_monthly_labels{VERSION}"
BASELINE_MEAN_FOLDER   = f"{ASSET_ROOT}/seasonal_baseline_mean{VERSION}"
BASELINE_MEDIAN_FOLDER = f"{ASSET_ROOT}/seasonal_baseline_median{VERSION}"
BASELINE_STD_FOLDER    = f"{ASSET_ROOT}/seasonal_baseline_std{VERSION}"
ANOMALY_FOLDER         = f"{ASSET_ROOT}/monthly_anomaly{VERSION}"

# Source assets (region of interest, masks, fire ground truth)
ROI_FC_ID         = f"{ASSET_ROOT}/gaza_envelope_for_decell_Dissolve"
LANDUSE_FC_ID     = f"{ASSET_ROOT}/otef_landuse"
HUMAN_INFRA_FC_ID = f"{ASSET_ROOT}/human_infrastructure_mask_utm36n"
FIRES_TRUTH_FC_ID = f"{ASSET_ROOT}/fires_scars_otef_6991"

print("=" * 60)
print(f"VERSION    = '{VERSION}'")
print(f"Resolution = {SCALE} m")
print("=" * 60)
print(f"S2 composites    -> {ASSET_COLLECTION}")
print(f"Fire labels      -> {LABEL_COLLECTION}")
print(f"Baseline mean    -> {BASELINE_MEAN_FOLDER}")
print(f"Baseline median  -> {BASELINE_MEDIAN_FOLDER}")
print(f"Baseline std     -> {BASELINE_STD_FOLDER}")
print(f"Anomaly          -> {ANOMALY_FOLDER}")
print("=" * 60)

VERSION    = '_10m'
Resolution = 10 m
S2 composites    -> projects/bar-project-457815/assets/s2_monthly_image_collection_10m
Fire labels      -> projects/bar-project-457815/assets/fire_monthly_labels_10m
Baseline mean    -> projects/bar-project-457815/assets/seasonal_baseline_mean_10m
Baseline median  -> projects/bar-project-457815/assets/seasonal_baseline_median_10m
Baseline std     -> projects/bar-project-457815/assets/seasonal_baseline_std_10m
Anomaly          -> projects/bar-project-457815/assets/monthly_anomaly_10m


In [9]:
# ============================================================
# Install dependencies and initialize Earth Engine
# ============================================================

!pip install -q -U geemap

import ee
import geemap

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized for project:", PROJECT_ID)

Earth Engine initialized for project: bar-project-457815


In [10]:
# ============================================================
# Pre-flight check
# Lists target asset paths and reports whether each one
# already exists in the project.
# ============================================================

paths_to_check = [
    ASSET_COLLECTION,
    LABEL_COLLECTION,
    BASELINE_MEAN_FOLDER,
    BASELINE_MEDIAN_FOLDER,
    BASELINE_STD_FOLDER,
    ANOMALY_FOLDER,
]

print(f"Checking target paths for VERSION='{VERSION}':\n")

collisions = []
for path in paths_to_check:
    try:
        info = ee.data.getAsset(path)
        n_imgs = "?"
        try:
            n_imgs = ee.ImageCollection(path).size().getInfo()
        except Exception:
            pass
        print(f"  EXISTS  {path}  (type={info['type']}, n_images={n_imgs})")
        collisions.append(path)
    except Exception:
        print(f"  FREE    {path}")

print()
if collisions:
    print(f"{len(collisions)} target path(s) already exist.")
else:
    print(f"All target paths are free.")

Checking target paths for VERSION='_10m':

  FREE    projects/bar-project-457815/assets/s2_monthly_image_collection_10m
  FREE    projects/bar-project-457815/assets/fire_monthly_labels_10m
  FREE    projects/bar-project-457815/assets/seasonal_baseline_mean_10m
  FREE    projects/bar-project-457815/assets/seasonal_baseline_median_10m
  FREE    projects/bar-project-457815/assets/seasonal_baseline_std_10m
  FREE    projects/bar-project-457815/assets/monthly_anomaly_10m

All target paths are free.


In [11]:
# ============================================================
# Step 1 — Load the region of interest, spatial masks, and
# the fire ground truth polygons.
#
# - landuse_mask:     excludes pixels with FCODE == 501
# - human_infra_mask: excludes pixels covered by built infrastructure
# - final_mask:       intersection of both, applied to all imagery
# - fires_clean:      fire polygons within the analysis date range
# ============================================================

working_area = ee.FeatureCollection(ROI_FC_ID)
region       = working_area.geometry()

landuse        = ee.FeatureCollection(LANDUSE_FC_ID)
human_infra_fc = ee.FeatureCollection(HUMAN_INFRA_FC_ID)

landuse_mask = (landuse
                .filter(ee.Filter.neq("FCODE", 501))
                .reduceToImage(["FCODE"], ee.Reducer.first())
                .gt(0))

human_infra_mask = (human_infra_fc
                    .reduceToImage(["Shape_Area"], ee.Reducer.count())
                    .gt(0))

final_mask = landuse_mask.And(human_infra_mask.Not())

fires_truth = (ee.FeatureCollection(FIRES_TRUTH_FC_ID)
               .filterBounds(region)
               .map(lambda f: f.set("geom_type", f.geometry().type()))
               .filter(ee.Filter.inList("geom_type", ["Polygon", "MultiPolygon"]))
               .map(lambda f: f.set({
                    "fire_date":         ee.Date(ee.Number(f.get("FireDate"))),
                    "fire_year":         ee.Date(ee.Number(f.get("FireDate"))).get("year"),
                    "system:time_start": ee.Number(f.get("FireDate"))
               })))

START = ee.Date(DATE_START).millis()
END   = ee.Date(DATE_END).millis()

fires_clean = fires_truth.filter(ee.Filter.rangeContains("FireDate", START, END))

print("Fire polygons within date range:", fires_clean.size().getInfo())

# Binary raster: 1 if pixel falls inside any fire polygon across the full period
burned_any = ee.Image(0).toByte().paint(fires_clean, 1).rename("burned_any")

Fire polygons within date range: 2439


In [12]:
# ============================================================
# Step 2 — Build a clean Sentinel-2 collection.
#
# Bands include the 10 m visible/NIR bands and the 20 m
# red-edge and SWIR bands. Cloud Score+ (cs_cdf) is used
# to mask cloudy pixels at a threshold of CLEAR_THRESH.
# The spatial mask from Step 1 is applied to every image.
# ============================================================

BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(region)
      .filterDate(DATE_START, DATE_END)
      .select(BANDS))

csplus = (ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
          .filterBounds(region)
          .filterDate(DATE_START, DATE_END))

# Link cs_cdf score from Cloud Score+ onto each Sentinel-2 image
s2_with_cs = s2.linkCollection(csplus, ["cs_cdf"])

def mask_with_cs(img):
    return (img
            .updateMask(img.select("cs_cdf").gte(CLEAR_THRESH))
            .select(BANDS)
            .updateMask(final_mask)
            .toFloat())

s2_clean = ee.ImageCollection(s2_with_cs.map(mask_with_cs))

print("Clean Sentinel-2 images over ROI:", s2_clean.size().getInfo())

# Verify the native projection of a SWIR band
img  = s2_clean.first()
proj = img.select("B11").projection()
print("CRS:      ", proj.crs().getInfo())
print("Transform:", proj.transform().getInfo())

Clean Sentinel-2 images over ROI: 1305
CRS:       EPSG:32636
Transform: PARAM_MT["Affine", 
  PARAMETER["num_row", 3], 
  PARAMETER["num_col", 3], 
  PARAMETER["elt_0_0", 20.0], 
  PARAMETER["elt_0_2", 600000.0], 
  PARAMETER["elt_1_1", -20.0], 
  PARAMETER["elt_1_2", 3500040.0]]


In [14]:
# ============================================================
# Step 3 — Build monthly mean composites with 13 vegetation
# and burn indices, plus two quality bands (n_obs_valid and
# pct_valid). Each month is exported as a separate Earth
# Engine image asset.
#
# Indices included:
#   Vegetation:  NDVI, EVI, SAVI, NDRE
#   Moisture:    NDMI, MSI, NDWI, MNDWI
#   Burn/SWIR:   NBR, NBR2, MIRBI, BSI, BAIS2
# ============================================================

INDICES = [
    "NDVI", "EVI", "SAVI", "NDRE",
    "NDMI", "MSI",  "NDWI", "MNDWI",
    "NBR",  "NBR2", "MIRBI", "BSI",
    "BAIS2"
]

def add_indices(img_ref):
    eps = ee.Image.constant(1e-6)

    b2  = img_ref.select("B2")
    b3  = img_ref.select("B3")
    b4  = img_ref.select("B4")
    b5  = img_ref.select("B5")
    b8  = img_ref.select("B8")
    b8a = img_ref.select("B8A")
    b11 = img_ref.select("B11")
    b12 = img_ref.select("B12")

    # Core normalized indices
    ndvi = img_ref.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = img_ref.normalizedDifference(["B8", "B11"]).rename("NDMI")
    nbr  = img_ref.normalizedDifference(["B8", "B12"]).rename("NBR")

    # Enhanced and soil-adjusted vegetation indices
    evi = (b8.subtract(b4).multiply(2.5)
             .divide(b8.add(b4.multiply(6)).subtract(b2.multiply(7.5)).add(1).add(eps))
             .rename("EVI"))

    L = ee.Image.constant(0.5)
    savi = (b8.subtract(b4).multiply(L.add(1))
              .divide(b8.add(b4).add(L).add(eps))
              .rename("SAVI"))

    ndre = (b8a.subtract(b5)
              .divide(b8a.add(b5).add(eps))
              .rename("NDRE"))

    # SWIR-based burn and soil indices
    nbr2  = (b11.subtract(b12).divide(b11.add(b12).add(eps)).rename("NBR2"))
    msi   = (b11.divide(b8.add(eps))).rename("MSI")
    mirbi = (b12.multiply(10).subtract(b11.multiply(9.8)).add(2)).rename("MIRBI")
    bsi   = ((b11.add(b4).subtract(b8.add(b2)))
             .divide(b11.add(b4).add(b8).add(b2).add(eps))
             .rename("BSI"))

    # Water indices
    ndwi  = (b3.subtract(b8)).divide(b3.add(b8).add(eps)).rename("NDWI")
    mndwi = (b3.subtract(b11)).divide(b3.add(b11).add(eps)).rename("MNDWI")

    # BAIS2 (Burned Area Index for Sentinel-2)
    term1 = ee.Image(1).subtract(b11.multiply(b12).multiply(b8a).divide(b4.add(eps)).sqrt())
    term2 = b12.subtract(b8a).divide(b12.add(b8a).add(eps).sqrt()).add(1)
    bais2 = term1.multiply(term2).rename("BAIS2")

    return img_ref.addBands([
        ndvi, evi, savi, ndre,
        ndmi, msi, ndwi, mndwi,
        nbr, nbr2, mirbi, bsi,
        bais2
    ])

def monthly_mean_image(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")

    ic = s2_clean.filterDate(start, end)

    # Scale Sentinel-2 SR to surface reflectance in [0, 1] and append indices
    ic_ref = ic.map(lambda img:
                    img.select(BANDS).multiply(0.0001)
                       .copyProperties(img, img.propertyNames()))
    ic_ref = ic_ref.map(add_indices)

    comp = ic_ref.select(BANDS + INDICES).mean()

    # Quality bands: number of valid observations and fraction relative to total month coverage
    n_obs_valid       = ic_ref.select("B4").count().rename("n_obs_valid")
    n_images_in_month = ee.Number(ic_ref.size())
    pct_valid         = n_obs_valid.divide(n_images_in_month).rename("pct_valid")

    out = (comp
           .addBands([n_obs_valid, pct_valid])
           .set({
               "year":              year,
               "month":             month,
               "month_start":       start.format("YYYY-MM-dd"),
               "n_images_in_month": n_images_in_month,
               "system:time_start": start.millis()
           })
           .clip(region))
    return out

def export_month_to_asset(year, month):
    img      = monthly_mean_image(year, month)
    ym       = f"{int(year)}_{int(month):02d}"
    desc     = f"s2_monthly_mean_{ym}"
    asset_id = f"{ASSET_COLLECTION}/{desc}"

    task = ee.batch.Export.image.toAsset(
        image            = img,
        description      = desc,
        assetId          = asset_id,
        region           = region,
        scale            = SCALE,
        crs              = CRS,
        maxPixels        = MAX_PIXELS,
        pyramidingPolicy = {'default': 'mean'}
    )
    task.start()
    return task

# Submit export tasks for every month of the analysis period
tasks = []
for y in range(2017, 2026):
    for m in range(1, 13):
        tasks.append(export_month_to_asset(y, m))

print(f"Submitted {len(tasks)} monthly composite export tasks")
print(f"Target collection: {ASSET_COLLECTION}")

Submitted 108 monthly composite export tasks
Target collection: projects/bar-project-457815/assets/s2_monthly_image_collection_10m


Finish running "phase 1". wait to complete and then run the rest:

In [ ]:
# ============================================================
# Step 4 — Build monthly binary fire-label images from the
# FireDate fire-scar polygons and export each month as a
# separate image asset.
#
# Each pixel is 1 if any fire polygon active in that month
# overlaps it, and 0 otherwise.
#
# Run only after the Step 3 export tasks have completed.
# ============================================================

def monthly_fire_label_image(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")

    fires_m = fires_clean.filterDate(start, end)

    label = (ee.Image(0).toByte()
             .paint(fires_m, 1)
             .rename("burned_this_month")
             .clip(region)
             .unmask(0)
             .set({
                 "year":              year,
                 "month":             month,
                 "month_start":       start.format("YYYY-MM-dd"),
                 "n_fire_polygons":   fires_m.size(),
                 "system:time_start": start.millis()
             }))
    return label

def export_monthly_fire_label_to_asset(year, month):
    img      = monthly_fire_label_image(year, month)
    ym       = f"{int(year)}_{int(month):02d}"
    desc     = f"fire_label_{ym}"
    asset_id = f"{LABEL_COLLECTION}/{desc}"

    task = ee.batch.Export.image.toAsset(
        image       = img,
        description = desc,
        assetId     = asset_id,
        region      = region,
        scale       = SCALE,
        crs         = CRS,
        maxPixels   = MAX_PIXELS
    )
    task.start()
    return task

tasks = []
for y in range(2017, 2026):
    for m in range(1, 13):
        tasks.append(export_monthly_fire_label_to_asset(y, m))

print(f"Submitted {len(tasks)} fire-label export tasks")
print(f"Target collection: {LABEL_COLLECTION}")

In [ ]:
# ============================================================
# Step 5 — Compute per-calendar-month baseline statistics
# (mean, median, standard deviation) from the 2017-2019
# reference period, using the monthly composite collection
# exported in Step 3. The three baseline image collections
# are required for the Z-score anomaly in Step 6.
#
# Run only after the Step 3 export tasks have completed.
# ============================================================

ALL_BANDS = [
    "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12",
    "NDVI", "EVI", "SAVI", "NDRE", "NDMI", "MSI",
    "NDWI", "MNDWI", "NBR", "NBR2", "MIRBI", "BSI", "BAIS2",
    "n_obs_valid", "pct_valid"
]

ic_full = ee.ImageCollection(ASSET_COLLECTION)

# Reference period for baseline computation
ic_ref = ic_full.filter(
    ee.Filter.And(
        ee.Filter.gte("year", 2017),
        ee.Filter.lte("year", 2019)
    )
)
print("Images in reference period (2017-2019):", ic_ref.size().getInfo())

baseline_tasks = []

for month in range(1, 13):
    ic_month = ic_ref.filter(ee.Filter.eq("month", month))
    n = ic_month.size().getInfo()
    print(f"  Month {month:02d}: {n} images")

    # Per-band mean across reference years for this calendar month
    baseline_mean = (ic_month.select(ALL_BANDS).mean()
                     .rename(ALL_BANDS)
                     .set({
                         "month":             month,
                         "baseline_type":     "mean",
                         "reference_years":   "2017-2019",
                         "system:time_start": ee.Date.fromYMD(2018, month, 1).millis()
                     })
                     .clip(region))

    # Per-band median across reference years for this calendar month
    baseline_median = (ic_month.select(ALL_BANDS).median()
                       .rename(ALL_BANDS)
                       .set({
                           "month":             month,
                           "baseline_type":     "median",
                           "reference_years":   "2017-2019",
                           "system:time_start": ee.Date.fromYMD(2018, month, 1).millis()
                       })
                       .clip(region))

    # Per-band standard deviation across reference years for this calendar month
    baseline_std = (ic_month.select(ALL_BANDS)
                    .reduce(ee.Reducer.stdDev())
                    .rename([b + "_stdDev" for b in ALL_BANDS])
                    .rename(ALL_BANDS)
                    .set({
                        "month":             month,
                        "baseline_type":     "std",
                        "reference_years":   "2017-2019",
                        "system:time_start": ee.Date.fromYMD(2018, month, 1).millis()
                    })
                    .clip(region))

    mm = f"{month:02d}"

    t1 = ee.batch.Export.image.toAsset(
        image       = baseline_mean,
        description = f"baseline_mean_{mm}",
        assetId     = f"{BASELINE_MEAN_FOLDER}/baseline_mean_{mm}",
        region      = region, scale = SCALE, crs = CRS, maxPixels = MAX_PIXELS
    )
    t1.start()

    t2 = ee.batch.Export.image.toAsset(
        image       = baseline_median,
        description = f"baseline_median_{mm}",
        assetId     = f"{BASELINE_MEDIAN_FOLDER}/baseline_median_{mm}",
        region      = region, scale = SCALE, crs = CRS, maxPixels = MAX_PIXELS
    )
    t2.start()

    t3 = ee.batch.Export.image.toAsset(
        image       = baseline_std,
        description = f"baseline_std_{mm}",
        assetId     = f"{BASELINE_STD_FOLDER}/baseline_std_{mm}",
        region      = region, scale = SCALE, crs = CRS, maxPixels = MAX_PIXELS
    )
    t3.start()

    baseline_tasks.extend([t1, t2, t3])

print(f"\nSubmitted {len(baseline_tasks)} baseline export tasks")
print(f"  Mean:   {BASELINE_MEAN_FOLDER}")
print(f"  Median: {BASELINE_MEDIAN_FOLDER}")
print(f"  Std:    {BASELINE_STD_FOLDER}")

In [ ]:
# ============================================================
# Step 6 — Compute monthly anomalies relative to the
# seasonal baseline. For each (year, month) image, three
# representations are exported:
#
#   Z-score anomaly = (observed - baseline_mean) / baseline_std
#   Raw anomaly     = observed - baseline_median
#   dNBR            = baseline_median(NBR) - observed(NBR)
#
# Z-score and raw anomaly are computed on seven indices used
# downstream: NDVI, NBR, MIRBI, NDMI, BSI, BAIS2, EVI.
#
# dNBR follows the convention of Key & Benson (2006) adapted
# to a seasonal-baseline reference: a positive value indicates
# vegetation loss relative to the pixel's typical condition
# for that calendar month.
#
# Run only after the Step 5 export tasks have completed.
# ============================================================

ANOMALY_BANDS = ["NDVI", "NBR", "MIRBI", "NDMI", "BSI", "BAIS2", "EVI"]

ic_full   = ee.ImageCollection(ASSET_COLLECTION)
ic_mean   = ee.ImageCollection(BASELINE_MEAN_FOLDER)
ic_median = ee.ImageCollection(BASELINE_MEDIAN_FOLDER)
ic_std    = ee.ImageCollection(BASELINE_STD_FOLDER)

def get_baseline(ic, month, bands):
    return ee.Image(
        ic.filter(ee.Filter.eq("month", month)).first()
    ).select(bands)

# All (year, month) pairs in the analysis period
MONTHS = []
for y in range(2017, 2026):
    for m in range(1, 13):
        if y == 2017 and m == 1:
            continue
        MONTHS.append((y, m))

anomaly_tasks = []

for (year, month) in MONTHS:
    ym = f"{year}_{month:02d}"

    observed = (ic_full
                .filter(ee.Filter.eq("year",  year))
                .filter(ee.Filter.eq("month", month))
                .first())

    observed = ee.Image(observed).select(ANOMALY_BANDS).unmask(-9999)

    b_mean   = get_baseline(ic_mean,   month, ANOMALY_BANDS)
    b_median = get_baseline(ic_median, month, ANOMALY_BANDS)
    b_std    = get_baseline(ic_std,    month, ANOMALY_BANDS)

    eps = ee.Image.constant(1e-6)

    z_score = (observed
               .subtract(b_mean)
               .divide(b_std.add(eps))
               .rename([b + "_zscore" for b in ANOMALY_BANDS]))

    raw_anomaly = (observed
                   .subtract(b_median)
                   .rename([b + "_anomaly" for b in ANOMALY_BANDS]))

    # dNBR relative to the seasonal baseline median
    # Sign convention: positive dNBR = vegetation loss
    dnbr = (b_median.select("NBR")
            .subtract(observed.select("NBR"))
            .rename("NBR_dnbr"))

    # Final image: 7 z-score + 7 raw anomaly + 1 dNBR = 15 bands
    anomaly_img = (z_score
                   .addBands(raw_anomaly)
                   .addBands(dnbr)
                   .set({
                       "year":              year,
                       "month":             month,
                       "system:time_start": ee.Date.fromYMD(year, month, 1).millis()
                   })
                   .clip(region))

    task = ee.batch.Export.image.toAsset(
        image       = anomaly_img,
        description = f"anomaly_{ym}",
        assetId     = f"{ANOMALY_FOLDER}/anomaly_{ym}",
        region      = region,
        scale       = SCALE,
        crs         = CRS,
        maxPixels   = MAX_PIXELS
    )
    task.start()
    anomaly_tasks.append(task)

print(f"Submitted {len(anomaly_tasks)} anomaly export tasks")
print(f"Target collection: {ANOMALY_FOLDER}")
print(f"Bands per image:   {len(ANOMALY_BANDS) * 2 + 1} (7 z-score + 7 raw anomaly + 1 dNBR)")

In [ ]:
# ----------------------------------------
# Best-practice tabular dataset for fire detection and severity proxy
# Sentinel-2 SR + Cloud Score+ + NDVI/NBR + dNBR + RBR
# Includes neighborhood mean and std (default 3x3)
# Balanced burned vs background, matched by date and local area
# ----------------------------------------

!pip install -U geemap
import ee, geemap

ee.Authenticate()
ee.Initialize(project='bar-project-457815')

ASSET_COLLECTION = "projects/bar-project-457815/assets/s2_monthly_image_collection"

In [ ]:
# -------------------------------------------------------
# 1) Load ROI, masks, and fire ground truth
# -------------------------------------------------------
working_area = ee.FeatureCollection("projects/bar-project-457815/assets/gaza_envelope_for_decell_Dissolve")
region = working_area.geometry()

landuse = ee.FeatureCollection("projects/bar-project-457815/assets/otef_landuse")
human_infra_fc = ee.FeatureCollection("projects/bar-project-457815/assets/human_infrastructure_mask_utm36n")

landuse_mask = (landuse
                .filter(ee.Filter.neq("FCODE", 501))
                .reduceToImage(["FCODE"], ee.Reducer.first())
                .gt(0))

human_infra_mask = (human_infra_fc
                    .reduceToImage(["Shape_Area"], ee.Reducer.count())
                    .gt(0))

final_mask = landuse_mask.And(human_infra_mask.Not())
fires_truth = (ee.FeatureCollection("projects/bar-project-457815/assets/fires_scars_otef_6991")
               .filterBounds(region)
               .map(lambda f: f.set("geom_type", f.geometry().type()))
               .filter(ee.Filter.inList("geom_type", ["Polygon", "MultiPolygon"]))
               .map(lambda f: f.set({
                    "fire_date": ee.Date(ee.Number(f.get("FireDate"))),
                    "fire_year": ee.Date(ee.Number(f.get("FireDate"))).get("year"),
                    "system:time_start": ee.Number(f.get("FireDate"))
                })))


START = ee.Date("2017-01-01").millis()
END   = ee.Date("2025-12-31").millis()
fires_clean = fires_truth.filter(ee.Filter.rangeContains("FireDate", START, END))

print("Fire polygons (clean):", fires_clean.size().getInfo())

# A single raster to test "is this pixel inside ANY fire polygon? (from all over the years)"
burned_any = ee.Image(0).toByte().paint(fires_clean, 1).rename("burned_any")

# -------------------------------------------------------
# 2) Load Sentinel-2 SR + Cloud Score+ and build clean collection
# Use 20m scale because SWIR/red-edge are 20m (good practice for NBR & red-edge indices)
# -------------------------------------------------------
DATE_START = "2017-01-01"
DATE_END   = "2025-12-31"

SCALE = 10
CLEAR_THRESH = 0.60

# Core + red-edge + SWIR
BANDS = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(region)
      .filterDate(DATE_START, DATE_END)
      .select(BANDS))

csplus = (ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
          .filterBounds(region)
          .filterDate(DATE_START, DATE_END))

# Attach cs_cdf to S2 images
s2_with_cs = s2.linkCollection(csplus, ["cs_cdf"])

def mask_with_cs(img):
    return (img
            .updateMask(img.select("cs_cdf").gte(CLEAR_THRESH))
            .select(BANDS)
            .updateMask(final_mask)
            .toFloat())

s2_clean = ee.ImageCollection(s2_with_cs.map(mask_with_cs))
print("Clean Sentinel-2 images over ROI:", s2_clean.size().getInfo())


#todo- save as assets to my project- for each month for all the reaerch area, with the indexes and the wave length i need. (montlhy image collection as asset)
# לטעון את הצורות על הראסטרים האלו ולהגיד לפיקסל שיקבל עוד טבלה אחת שהיא עושה תוית של נשרף או לא נשרף (אחד או 0) האם השריפה קרתה בפיקסל המסוים בחודש המסוים בשנה המסוימת
# ואז אפשר לשאול שאלות למשל מה הפיקסל שנשרף הכי הרבה פעמים וכו. יש לי פה סטים של תמונות לאורך זמן ועליהן אוכל לתשאל


NameError: name 'ee' is not defined

In [ ]:
img = s2_clean.first()
proj = img.select("B11").projection()
print("Projection:", proj.getInfo())
print("CRS:", proj.crs().getInfo())
print("Transform:", proj.transform().getInfo())

In [ ]:
# -------------------------------------------------------
# 3) Build MONTHLY MEAN composites (clean S2) + more indices + quality
#    and export EACH MONTH as an Image Asset to your folder
# -------------------------------------------------------

ASSET_FOLDER = "projects/bar-project-457815/assets/s2_monthly_image_collection"
CRS = "EPSG:32636"
SCALE = 10
MAX_PIXELS = 1e13

# Keep your current bands (10m + 20m); exported at 20m -> 10m bands will be resampled
BANDS = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]

# Indices to include in the monthly asset
INDICES = [
    "NDVI","EVI","SAVI","NDRE",
    "NDMI","MSI","NDWI","MNDWI",
    "NBR","NBR2","MIRBI","BSI",
    "BAIS2"
]

def add_indices(img_ref):
    eps = ee.Image.constant(1e-6)

    b2  = img_ref.select("B2")   # Blue
    b3  = img_ref.select("B3")   # Green
    b4  = img_ref.select("B4")   # Red
    b5  = img_ref.select("B5")   # Red-edge 1
    b8  = img_ref.select("B8")   # NIR
    b8a = img_ref.select("B8A")  # NIR narrow
    b11 = img_ref.select("B11")  # SWIR1
    b12 = img_ref.select("B12")  # SWIR2

    # Core
    ndvi = img_ref.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = img_ref.normalizedDifference(["B8", "B11"]).rename("NDMI")
    nbr  = img_ref.normalizedDifference(["B8", "B12"]).rename("NBR")

    # Vegetation robust
    evi = (b8.subtract(b4)
           .multiply(2.5)
           .divide(b8.add(b4.multiply(6)).subtract(b2.multiply(7.5)).add(1).add(eps))
           .rename("EVI"))

    L = ee.Image.constant(0.5)
    savi = (b8.subtract(b4)
            .multiply(L.add(1))
            .divide(b8.add(b4).add(L).add(eps))
            .rename("SAVI"))

    ndre = (b8a.subtract(b5)
            .divide(b8a.add(b5).add(eps))
            .rename("NDRE"))

    # Burn / SWIR focused
    nbr2 = (b11.subtract(b12)
            .divide(b11.add(b12).add(eps))
            .rename("NBR2"))

    msi = (b11.divide(b8.add(eps))).rename("MSI")  # SWIR1 / NIR

    mirbi = (b12.multiply(10).subtract(b11.multiply(9.8)).add(2)).rename("MIRBI")

    bsi = ((b11.add(b4).subtract(b8.add(b2)))
           .divide(b11.add(b4).add(b8).add(b2).add(eps))
           .rename("BSI"))

    # Water-related (helps avoid confusion with wet soil/irrigation)
    ndwi  = (b3.subtract(b8)).divide(b3.add(b8).add(eps)).rename("NDWI")      # McFeeters
    mndwi = (b3.subtract(b11)).divide(b3.add(b11).add(eps)).rename("MNDWI")   # Xu

    # BAIS2 (your original idea)
    term1 = ee.Image(1).subtract(b11.multiply(b12).multiply(b8a).divide(b4.add(eps)).sqrt())
    term2 = b12.subtract(b8a).divide(b12.add(b8a).add(eps).sqrt()).add(1)
    bais2 = term1.multiply(term2).rename("BAIS2")

    return img_ref.addBands([
        ndvi, evi, savi, ndre,
        ndmi, msi, ndwi, mndwi,
        nbr, nbr2, mirbi, bsi,
        bais2
    ])

def monthly_mean_image(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")

    ic = s2_clean.filterDate(start, end)

    # Convert S2_SR to reflectance [0..1] for stable index math
    ic_ref = ic.map(lambda img: img.select(BANDS).multiply(0.0001).copyProperties(img, img.propertyNames()))
    ic_ref = ic_ref.map(add_indices)

    # Monthly mean composite (per pixel)
    comp = ic_ref.select(BANDS + INDICES).mean()

    # Quality bands
    n_obs_valid = ic_ref.select("B4").count().rename("n_obs_valid")  # per-pixel valid obs count
    n_images_in_month = ee.Number(ic_ref.size())                     # scalar
    pct_valid = n_obs_valid.divide(n_images_in_month).rename("pct_valid")

    out = (comp
           .addBands([n_obs_valid, pct_valid])
           .set({
               "year": year,
               "month": month,
               "month_start": start.format("YYYY-MM-dd"),
               "n_images_in_month": n_images_in_month,
               "system:time_start": start.millis()
           })
           .clip(region))

    return out

def export_month_to_asset(year, month):
    img = monthly_mean_image(year, month)
    ym = f"{int(year)}_{int(month):02d}"
    desc = f"s2_monthly_mean_{ym}"
    asset_id = f"{ASSET_COLLECTION}/{desc}"

    task = ee.batch.Export.image.toAsset(
        image=img,
        description=desc,
        assetId=asset_id,
        region=region,
        scale=SCALE,
        crs=CRS,
        maxPixels=MAX_PIXELS,
        pyramidingPolicy={'default': 'mean'}
    )
    task.start()
    return task

# -------------------------------------------------------
# Run exports (start small, then scale up)
# -------------------------------------------------------

# Run for all the years: 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025
tasks = []
for y in [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]:
    for m in range(1, 13):
        tasks.append(export_month_to_asset(y, m))

print(f"{len(tasks)} exported tasks for year 2017,2018,2019,2020,2021,2022,2023,2024,2025")

# When you're happy, export all years:
# tasks = []
# for y in range(2017, 2026):
#     for m in range(1, 13):
#         tasks.append(export_month_to_asset(y, m))
# print(f"Started {len(tasks)} export tasks (2017-2025).")


NameError: name 'ee' is not defined

In [ ]:
# -------------------------------------------------------
# 4) Build MONTHLY FIRE LABELS (0/1) from FireDate polygons
#    and export EACH MONTH as an Image Asset
# -------------------------------------------------------

LABEL_COLLECTION = "projects/bar-project-457815/assets/fire_monthly_labels"

CRS = "EPSG:32636"
SCALE = 10
MAX_PIXELS = 1e13

def monthly_fire_label_image(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")

    # Now that your features have system:time_start, this is the best way:
    fires_m = fires_clean.filterDate(start, end)

    label = (ee.Image(0).toByte()
             .paint(fires_m, 1)
             .rename("burned_this_month")
             .clip(region)
             .unmask(0)
             .set({
                 "year": year,
                 "month": month,
                 "month_start": start.format("YYYY-MM-dd"),
                 "n_fire_polygons": fires_m.size(),
                 "system:time_start": start.millis()
             }))
    return label

def export_monthly_fire_label_to_asset(year, month):
    img = monthly_fire_label_image(year, month)

    ym = f"{int(year)}_{int(month):02d}"
    desc = f"fire_label_{ym}"
    asset_id = f"{LABEL_COLLECTION}/{desc}"

    task = ee.batch.Export.image.toAsset(
        image=img,
        description=desc,
        assetId=asset_id,
        region=region,
        scale=SCALE,
        crs=CRS,
        maxPixels=MAX_PIXELS
    )
    task.start()
    return task

tasks = []
for y in [2017,2018,2019,2020,2021,2022,2023,2024, 2025]:
    for m in range(1, 13):
        tasks.append(export_monthly_fire_label_to_asset(y, m))

print(f"Started {len(tasks)} fire-label export tasks for year 2017, 2018,2019, 2020, 2021, 2022, 2023,2024,2025.")


NameError: name 'ee' is not defined

for the harmonic model:


In [ ]:
# ============================================================
# STEP: Compute seasonal baseline (mean, median, std)
#       from 2017-2019 reference period
#       and export monthly anomaly collection
# ============================================================

# ---- Asset paths ----
ASSET_COLLECTION      = "projects/bar-project-457815/assets/s2_monthly_image_collection"
BASELINE_MEAN_FOLDER  = "projects/bar-project-457815/assets/seasonal_baseline_mean"
BASELINE_MEDIAN_FOLDER= "projects/bar-project-457815/assets/seasonal_baseline_median"
BASELINE_STD_FOLDER   = "projects/bar-project-457815/assets/seasonal_baseline_std"
ANOMALY_FOLDER        = "projects/bar-project-457815/assets/monthly_anomaly"

CRS        = "EPSG:32636"
SCALE      = 10
MAX_PIXELS = 1e13

# Bands to include — same 25 as your composites
# (reflectance bands + indices + quality)
ALL_BANDS = [
    "B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12",
    "NDVI","EVI","SAVI","NDRE","NDMI","MSI",
    "NDWI","MNDWI","NBR","NBR2","MIRBI","BSI","BAIS2",
    "n_obs_valid","pct_valid"
]

# Load your full monthly composite collection
ic_full = ee.ImageCollection(ASSET_COLLECTION)

# ============================================================
# PART 1 — Compute baseline per calendar month (2017–2019)
# ============================================================
# Note: using 2017–2019 as reference period (least disturbed).
# median is preferred over mean — robust to the small fires
# that did ocur in those years.
# We save mean, median, AND std so we can compute Z-score anomalies.

# Filter to reference period only
ic_ref = ic_full.filter(
    ee.Filter.And(
        ee.Filter.gte("year", 2017),
        ee.Filter.lte("year", 2019)
    )
)

print("Images in reference period (2017–2019):", ic_ref.size().getInfo())

baseline_tasks = []

for month in range(1, 13):

    # All images from this calendar month across 2017–2019
    ic_month = ic_ref.filter(ee.Filter.eq("month", month))

    n = ic_month.size().getInfo()
    print(f"  Month {month:02d}: {n} images in reference period")

    # --- Mean ---
    baseline_mean = (ic_month
                     .select(ALL_BANDS)
                     .mean()
                     .rename(ALL_BANDS)
                     .set({
                         "month": month,
                         "baseline_type": "mean",
                         "reference_years": "2017-2019",
                         "system:time_start": ee.Date.fromYMD(2018, month, 1).millis()
                     })
                     .clip(region))

    # --- Median ---
    baseline_median = (ic_month
                       .select(ALL_BANDS)
                       .median()
                       .rename(ALL_BANDS)
                       .set({
                           "month": month,
                           "baseline_type": "median",
                           "reference_years": "2017-2019",
                           "system:time_start": ee.Date.fromYMD(2018, month, 1).millis()
                       })
                       .clip(region))

    # --- Std dev ---
    # GEE reducer: stdDev across the collection
    baseline_std = (ic_month
                    .select(ALL_BANDS)
                    .reduce(ee.Reducer.stdDev())
                    # reduce() appends "_stdDev" to band names — rename back
                    .rename([b + "_stdDev" for b in ALL_BANDS])
                    .rename(ALL_BANDS)
                    .set({
                        "month": month,
                        "baseline_type": "std",
                        "reference_years": "2017-2019",
                        "system:time_start": ee.Date.fromYMD(2018, month, 1).millis()
                    })
                    .clip(region))

    mm = f"{month:02d}"

    # Export mean
    t1 = ee.batch.Export.image.toAsset(
        image=baseline_mean,
        description=f"baseline_mean_{mm}",
        assetId=f"{BASELINE_MEAN_FOLDER}/baseline_mean_{mm}",
        region=region, scale=SCALE, crs=CRS, maxPixels=MAX_PIXELS
    )
    t1.start()

    # Export median
    t2 = ee.batch.Export.image.toAsset(
        image=baseline_median,
        description=f"baseline_median_{mm}",
        assetId=f"{BASELINE_MEDIAN_FOLDER}/baseline_median_{mm}",
        region=region, scale=SCALE, crs=CRS, maxPixels=MAX_PIXELS
    )
    t2.start()

    # Export std
    t3 = ee.batch.Export.image.toAsset(
        image=baseline_std,
        description=f"baseline_std_{mm}",
        assetId=f"{BASELINE_STD_FOLDER}/baseline_std_{mm}",
        region=region, scale=SCALE, crs=CRS, maxPixels=MAX_PIXELS
    )
    t3.start()

    baseline_tasks.extend([t1, t2, t3])

print(f"\nStarted {len(baseline_tasks)} baseline export tasks (12 mean + 12 median + 12 std)")
print("Wait for all tasks to complete in GEE Task Manager before running Part 2.")


NameError: name 'ee' is not defined

In [ ]:
# ============================================================
# PART 2 — Compute Z-score anomaly for each of the 106 months
#           Run this ONLY after Part 1 exports are complete
# ============================================================
# Z-score anomaly = (observed - baseline_mean) / baseline_std
# Saving only 7 key indices instead of all 25 — saves ~72% storage
# with no meaningful loss to model quality.
# Raw values of all 25 bands remain in s2_monthly_image_collection.

ASSET_COLLECTION      = "projects/bar-project-457815/assets/s2_monthly_image_collection"
BASELINE_MEAN_FOLDER  = "projects/bar-project-457815/assets/seasonal_baseline_mean"
BASELINE_MEDIAN_FOLDER= "projects/bar-project-457815/assets/seasonal_baseline_median"
BASELINE_STD_FOLDER   = "projects/bar-project-457815/assets/seasonal_baseline_std"
ANOMALY_FOLDER        = "projects/bar-project-457815/assets/monthly_anomaly"
CRS        = "EPSG:32636"
SCALE      = 10
MAX_PIXELS = 1e13

# *** CHANGED: only 7 key indices instead of all 25 ***
# NDVI  — vegetation greenness
# NBR   — NIR vs SWIR2, most burn-sensitive
# MIRBI — SWIR1 vs SWIR2, burn severity
# NDMI  — NIR vs SWIR1, moisture stress
# BSI   — bare soil exposure post-fire
# BAIS2 — Sentinel-2 specific burn index
# EVI   — atmospherically corrected vegetation
ANOMALY_BANDS = ["NDVI", "NBR", "MIRBI", "NDMI", "BSI", "BAIS2", "EVI"]

ic_full   = ee.ImageCollection(ASSET_COLLECTION)
ic_mean   = ee.ImageCollection(BASELINE_MEAN_FOLDER)
ic_median = ee.ImageCollection(BASELINE_MEDIAN_FOLDER)
ic_std    = ee.ImageCollection(BASELINE_STD_FOLDER)

# *** CHANGED: accepts bands as parameter ***
def get_baseline(ic, month, bands):
    return ee.Image(
        ic.filter(ee.Filter.eq("month", month)).first()
    ).select(bands)

anomaly_tasks = []

MONTHS = []
for y in range(2017, 2026):
    for m in range(1, 13):
        if y == 2017 and m == 1:
            continue
        MONTHS.append((y, m))

for (year, month) in MONTHS:
    ym = f"{year}_{month:02d}"

    observed = (ic_full
                .filter(ee.Filter.eq("year", year))
                .filter(ee.Filter.eq("month", month))
                .first())

    # *** CHANGED: select only ANOMALY_BANDS ***
    observed = ee.Image(observed).select(ANOMALY_BANDS).unmask(-9999)

    # *** CHANGED: pass ANOMALY_BANDS to get_baseline ***
    b_mean   = get_baseline(ic_mean,   month, ANOMALY_BANDS)
    b_median = get_baseline(ic_median, month, ANOMALY_BANDS)
    b_std    = get_baseline(ic_std,    month, ANOMALY_BANDS)

    eps = ee.Image.constant(1e-6)

    z_score = (observed
               .subtract(b_mean)
               .divide(b_std.add(eps))
               .rename([b + "_zscore" for b in ANOMALY_BANDS]))

    raw_anomaly = (observed
                   .subtract(b_median)
                   .rename([b + "_anomaly" for b in ANOMALY_BANDS]))

    # 7 z-score + 7 raw anomaly = 14 bands per image
    anomaly_img = (z_score
                   .addBands(raw_anomaly)
                   .set({
                       "year": year,
                       "month": month,
                       "system:time_start": ee.Date.fromYMD(
                           year, month, 1).millis()
                   })
                   .clip(region))

    task = ee.batch.Export.image.toAsset(
        image=anomaly_img,
        description=f"anomaly_{ym}",
        assetId=f"{ANOMALY_FOLDER}/anomaly_{ym}",
        region=region,
        scale=SCALE,
        crs=CRS,
        maxPixels=MAX_PIXELS
    )
    task.start()
    anomaly_tasks.append(task)

print(f"Started {len(anomaly_tasks)} anomaly export tasks")
print(f"Bands per image: {len(ANOMALY_BANDS)*2} (7 z-score + 7 raw anomaly)")
print(f"Estimated total storage: ~17 GB (vs ~125 GB with all 25 bands)")


NameError: name 'ee' is not defined

In [ ]:
# ============================================================
# Validation - checking that the anomaly pipeline makes sense
# before moving to sampling and model training.
#
# Core idea: pick three representative pixels and plot their
# full time series. If the Z-score drops when the label says
# "burned", the pipeline is working. If not, something is wrong
# with the baseline or the anomaly export.
#
# Three pixel types:
#   A - burned exactly once in 2023 (clearest test case)
#   B — never burned (should show stable seasonality, Z ~ 0)
#   C — burned the most times (should show repeated drops)
#
# Pixels are selected automatically from existing assets —
# less bias and more reproducible than manual coordinate picking.
#
# Quality filtering:
#   Months with pct_valid < 0.2 are flagged and excluded.
#   Validation showed these produce |Z-score| > 10, which is
#   physically implausible and caused by cloud-contaminated
#   composites rather than real surface changes.
# ============================================================

!pip -q install geemap matplotlib
import ee, geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ee.Authenticate()
ee.Initialize(project="bar-project-457815")

# --- asset paths ---
ASSET_COLLECTION   = "projects/bar-project-457815/assets/s2_monthly_image_collection"
LABEL_COLLECTION   = "projects/bar-project-457815/assets/fire_monthly_labels"
ANOMALY_COLLECTION = "projects/bar-project-457815/assets/monthly_anomaly"
ROI_FC_ID          = "projects/bar-project-457815/assets/gaza_envelope_for_decell_Dissolve"

# quality threshold — months below this are excluded from training
# determined empirically: pct_valid < 0.2 produces |Z-score| > 10
QUALITY_THRESHOLD = 0.2

roi_fc     = ee.FeatureCollection(ROI_FC_ID)
region     = roi_fc.geometry()
ic_s2      = ee.ImageCollection(ASSET_COLLECTION)
ic_labels  = ee.ImageCollection(LABEL_COLLECTION).select("burned_this_month")
ic_anomaly = ee.ImageCollection(ANOMALY_COLLECTION)

# rebuild the spatial mask — same logic as the rest of the pipeline
landuse        = ee.FeatureCollection("projects/bar-project-457815/assets/otef_landuse")
human_infra_fc = ee.FeatureCollection(
    "projects/bar-project-457815/assets/human_infrastructure_mask_utm36n")

landuse_mask     = (landuse
                    .filter(ee.Filter.neq("FCODE", 501))
                    .reduceToImage(["FCODE"], ee.Reducer.first())
                    .gt(0))
human_infra_mask = (human_infra_fc
                    .reduceToImage(["Shape_Area"], ee.Reducer.count())
                    .gt(0))
final_mask = landuse_mask.And(human_infra_mask.Not())

# -------------------------------------------------------
# Step 1 — auto-select the three validation pixels
# -------------------------------------------------------

burn_count = (ic_labels
              .sum()
              .rename("burn_count")
              .updateMask(final_mask)
              .clip(region))

burned_2023 = (ic_labels
               .filter(ee.Filter.eq("year", 2023))
               .sum().gt(0)
               .rename("burned_2023")
               .updateMask(final_mask)
               .clip(region))

mean_ndvi = ic_s2.select("NDVI").mean().rename("mean_ndvi")

# pixel A: burned exactly once, in 2023
mask_A = burn_count.eq(1).And(burned_2023.eq(1)).selfMask()
coords_A = (ee.FeatureCollection(
                mask_A.stratifiedSample(
                    numPoints=50, classBand="burn_count",
                    region=region, scale=10, seed=42, geometries=True)
            ).first().geometry().coordinates().getInfo())
lon_A, lat_A = coords_A[0], coords_A[1]
print(f"Pixel A (single burn 2023):   {lon_A:.5f}, {lat_A:.5f}")

# pixel B: never burned, with some vegetation (NDVI > 0.15)
mask_B = (burn_count.eq(0)
          .selfMask()
          .updateMask(mean_ndvi.gt(0.15))
          .rename("burn_count"))
coords_B = (ee.FeatureCollection(
                mask_B.stratifiedSample(
                    numPoints=50, classBand="burn_count",
                    region=region, scale=10, seed=99, geometries=True)
            ).first().geometry().coordinates().getInfo())
lon_B, lat_B = coords_B[0], coords_B[1]
print(f"Pixel B (never burned):       {lon_B:.5f}, {lat_B:.5f}")

# pixel C: top 5% by burn count — highest recurrence
p95 = (burn_count.reduceRegion(
    reducer=ee.Reducer.percentile([95]),
    geometry=region, scale=10,
    maxPixels=1e13, bestEffort=True
).getInfo().get("burn_count_p95", 5))
print(f"  95th percentile burn count: {p95}")

mask_C = burn_count.gte(ee.Number(p95)).selfMask().rename("burn_count")
coords_C = (ee.FeatureCollection(
                mask_C.stratifiedSample(
                    numPoints=50, classBand="burn_count",
                    region=region, scale=10, seed=7, geometries=True)
            ).first().geometry().coordinates().getInfo())
lon_C, lat_C = coords_C[0], coords_C[1]
print(f"Pixel C (highest recurrence): {lon_C:.5f}, {lat_C:.5f}")

PIXELS = {
    "A_single_burn": {
        "lon": lon_A, "lat": lat_A,
        "description": "Burned once (2023)",
        "color": "#E24B4A"
    },
    "B_never_burned": {
        "lon": lon_B, "lat": lat_B,
        "description": "Never burned",
        "color": "#1D9E75"
    },
    "C_repeat_burn": {
        "lon": lon_C, "lat": lat_C,
        "description": "Highest recurrence",
        "color": "#EF9F27"
    },
}

# -------------------------------------------------------
# Step 2 — extract time series for each pixel
# -------------------------------------------------------

def extract_ts(lon, lat, collection, bands, scale=10):
    point = ee.Geometry.Point([lon, lat])

    def sample(img):
        vals = img.select(bands).reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=scale
        )
        return ee.Feature(None,
                          vals.set("year",  img.get("year"))
                              .set("month", img.get("month")))

    df = geemap.ee_to_df(ee.FeatureCollection(collection.map(sample)))
    df = df.dropna(subset=["year", "month"])
    df["year"]  = df["year"].astype(int)
    df["month"] = df["month"].astype(int)
    df = df.sort_values(["year", "month"]).reset_index(drop=True)
    df["time_label"] = df.apply(
        lambda r: f"{int(r['year'])}-{int(r['month']):02d}", axis=1)
    return df


def extract_labels(lon, lat, scale=10):
    point = ee.Geometry.Point([lon, lat])

    def sample(img):
        vals = img.select("burned_this_month").reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=scale
        )
        return ee.Feature(None,
                          vals.set("year",  img.get("year"))
                              .set("month", img.get("month")))

    df = geemap.ee_to_df(ee.FeatureCollection(ic_labels.map(sample)))
    df = df.dropna(subset=["year", "month"])
    df["year"]  = df["year"].astype(int)
    df["month"] = df["month"].astype(int)
    return df.sort_values(["year", "month"]).reset_index(drop=True)


print("\nExtracting time series — ~1-2 min per pixel...")

pixel_data = {}
for name, info in PIXELS.items():
    print(f"  {name}...")

    df_s2   = extract_ts(info["lon"], info["lat"],
                         ic_s2, ["NDVI", "NBR", "pct_valid"])
    df_anom = extract_ts(info["lon"], info["lat"],
                         ic_anomaly,
                         ["NDVI_zscore", "NBR_zscore", "NDVI_anomaly"])
    df_lbl  = extract_labels(info["lon"], info["lat"])

    df = (df_s2
          .merge(df_anom[["year","month",
                           "NDVI_zscore","NBR_zscore","NDVI_anomaly"]],
                 on=["year","month"], how="left")
          .merge(df_lbl[["year","month","burned_this_month"]],
                 on=["year","month"], how="left"))

    pixel_data[name] = {"df": df, **info}
    print(f"    {len(df)} months total")

# -------------------------------------------------------
# Step 3 — validation plots
# -------------------------------------------------------

# figure 1: raw NDVI + Z-score side by side
# main check: do drops align with fire labels?

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle(
    "Validation — NDVI time series and Z-score anomaly\n"
    "Western Negev 2017–2025  |  Sentinel-2 monthly composites  |  10 m",
    fontsize=12, fontweight="bold", y=1.01
)

for row, (name, pdata) in enumerate(pixel_data.items()):
    df    = pdata["df"]
    color = pdata["color"]
    desc  = pdata["description"]

    ticks     = list(range(0, len(df), 6))
    tick_lbls = [df["time_label"].iloc[i] for i in ticks]

    # left: raw NDVI
    ax1 = axes[row, 0]
    ax1.plot(range(len(df)), df["NDVI"],
             color=color, linewidth=1.8, label="NDVI")

    for i, r in df.iterrows():
        if r.get("burned_this_month", 0) == 1:
            ax1.axvspan(i - 0.5, i + 0.5,
                        color="#E24B4A", alpha=0.25, linewidth=0)

    # mark low quality months (below exclusion threshold)
    low_q = df[df["pct_valid"] < QUALITY_THRESHOLD]
    if len(low_q):
        ax1.scatter(low_q.index, low_q["NDVI"],
                    color="black", s=40, zorder=5, marker="x",
                    linewidths=1.5,
                    label=f"excluded (pct_valid < {QUALITY_THRESHOLD})")

    ax1.set_title(f"{desc}  —  raw NDVI", fontsize=10)
    ax1.set_ylabel("NDVI")
    ax1.set_ylim(-0.1, 1.0)
    ax1.set_xticks(ticks)
    ax1.set_xticklabels(tick_lbls, rotation=45, fontsize=7)
    ax1.axhline(0, color="black", linewidth=0.5, linestyle="--")
    ax1.legend(fontsize=7)
    ax1.grid(True, alpha=0.3)

    # right: Z-score — clip display to [-10, 10] for readability
    # values outside this range are excluded months (artifacts)
    ax2 = axes[row, 1]
    zscore_clipped = df["NDVI_zscore"].clip(-10, 10)
    ax2.plot(range(len(df)), zscore_clipped,
             color=color, linewidth=1.8, label="NDVI Z-score")
    ax2.axhline(0,  color="black",  linewidth=1,   linestyle="-")
    ax2.axhline(-1, color="orange", linewidth=0.8, linestyle="--",
                label="Z = −1")
    ax2.axhline(-2, color="red",    linewidth=0.8, linestyle="--",
                label="Z = −2")

    for i, r in df.iterrows():
        if r.get("burned_this_month", 0) == 1:
            ax2.axvspan(i - 0.5, i + 0.5,
                        color="#E24B4A", alpha=0.25, linewidth=0)

    # mark excluded months on Z-score plot too
    if len(low_q):
        ax2.scatter(low_q.index,
                    zscore_clipped.iloc[low_q.index],
                    color="black", s=40, zorder=5, marker="x",
                    linewidths=1.5)

    fire_patch = mpatches.Patch(color="#E24B4A", alpha=0.25,
                                label="labeled burned")
    handles, lbls = ax2.get_legend_handles_labels()
    ax2.legend(handles=handles + [fire_patch],
               labels=lbls + ["labeled burned"], fontsize=7)

    ax2.set_title(f"{desc}  —  NDVI Z-score (clipped to ±10)", fontsize=10)
    ax2.set_ylabel("std deviations from baseline")
    ax2.set_ylim(-10, 10)
    ax2.set_xticks(ticks)
    ax2.set_xticklabels(tick_lbls, rotation=45, fontsize=7)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("validation_fig1_ndvi_zscore.png", dpi=150, bbox_inches="tight")
plt.show()

# figure 2: NBR Z-score — more burn-sensitive than NDVI

fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle(
    "Validation — NBR Z-score per pixel type\n"
    "NBR (SWIR-based) should respond more sharply to burn events than NDVI",
    fontsize=11, fontweight="bold"
)

for col, (name, pdata) in enumerate(pixel_data.items()):
    df    = pdata["df"]
    color = pdata["color"]
    ax    = axes2[col]

    nbr_clipped = df["NBR_zscore"].clip(-10, 10)
    ax.plot(range(len(df)), nbr_clipped, color=color, linewidth=1.8)
    ax.axhline(0,  color="black", linewidth=1,   linestyle="-")
    ax.axhline(-2, color="red",   linewidth=0.8, linestyle="--",
               label="Z = −2")

    for i, r in df.iterrows():
        if r.get("burned_this_month", 0) == 1:
            ax.axvspan(i - 0.5, i + 0.5,
                       color="#E24B4A", alpha=0.3, linewidth=0)

    ticks     = list(range(0, len(df), 6))
    tick_lbls = [df["time_label"].iloc[i] for i in ticks]
    ax.set_xticks(ticks)
    ax.set_xticklabels(tick_lbls, rotation=45, fontsize=7)
    ax.set_title(pdata["description"], fontsize=9)
    ax.set_ylabel("NBR Z-score (clipped to ±10)")
    ax.set_ylim(-10, 10)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("validation_fig2_nbr_zscore.png", dpi=150, bbox_inches="tight")
plt.show()

# figure 3: quality threshold analysis
# documents which months are excluded and why

fig3, ax3 = plt.subplots(figsize=(16, 4))
ax3.set_title(
    f"Validation — data quality (pct_valid) and exclusion threshold\n"
    f"Months with pct_valid < {QUALITY_THRESHOLD} are excluded from model training",
    fontsize=11
)

for name, pdata in pixel_data.items():
    df = pdata["df"]
    ax3.plot(range(len(df)), df["pct_valid"],
             color=pdata["color"], linewidth=1.2,
             label=pdata["description"])

    excl = df[df["pct_valid"] < QUALITY_THRESHOLD]
    ax3.scatter(excl.index, excl["pct_valid"],
                color=pdata["color"], s=80, zorder=5,
                marker="x", linewidths=2)

ax3.axhline(QUALITY_THRESHOLD, color="red", linewidth=1.5,
            linestyle="--",
            label=f"Exclusion threshold: {QUALITY_THRESHOLD}")
ax3.set_ylabel("pct_valid")
ax3.set_ylim(0, 1.1)

ref_df    = pixel_data["A_single_burn"]["df"]
ticks     = list(range(0, len(ref_df), 6))
tick_lbls = [ref_df["time_label"].iloc[i] for i in ticks]
ax3.set_xticks(ticks)
ax3.set_xticklabels(tick_lbls, rotation=45, fontsize=8)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("validation_fig3_data_quality.png", dpi=150, bbox_inches="tight")
plt.show()

# -------------------------------------------------------
# Step 4 — quality threshold summary for thesis methods
# -------------------------------------------------------

print("=" * 60)
print("QUALITY THRESHOLD ANALYSIS")
print(f"Exclusion threshold: pct_valid < {QUALITY_THRESHOLD}")
print("=" * 60)

all_excluded = []

for name, pdata in pixel_data.items():
    df      = pdata["df"]
    excl    = df[df["pct_valid"] < QUALITY_THRESHOLD]
    kept    = df[df["pct_valid"] >= QUALITY_THRESHOLD]

    print(f"\nPixel {name}:")
    print(f"  Total months:    {len(df)}")
    print(f"  Excluded:        {len(excl)} ({100*len(excl)/len(df):.1f}%)")
    print(f"  Retained:        {len(kept)} ({100*len(kept)/len(df):.1f}%)")

    if len(excl) > 0:
        for _, r in excl.iterrows():
            z = r.get("NDVI_zscore", float("nan"))
            print(f"    {int(r['year'])}-{int(r['month']):02d} | "
                  f"pct_valid={r['pct_valid']:.3f} | "
                  f"NDVI_zscore={z:.1f}")
        all_excluded.extend(excl["time_label"].tolist())

unique_excl = sorted(set(all_excluded))
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Excluded months: {unique_excl}")
print(f"Total affected:  {len(unique_excl)} out of 106 "
      f"({100*len(unique_excl)/106:.1f}% of time series)")
print("""
Rationale (for thesis methods section):
  Monthly composites with pct_valid < 0.2 are based on fewer
  than 20% cloud-free observations. Validation showed these
  months produce |Z-score| > 10, which is physically implausible
  for vegetation indices and indicates cloud contamination
  rather than real surface change. These months are flagged
  automatically via the pct_valid band and excluded from
  model training. No manual intervention is required.
""")

# -------------------------------------------------------
# Step 5 — final checklist
# -------------------------------------------------------
print("""
Expected results if pipeline is correct:

Pixel A (single burn, 2023):
  - NDVI drops in the labeled fire month
  - Z-score drops below -1 or -2 at that point
  - Z-score gradually returns toward 0 afterward (recovery)
  - NBR drop is sharper and faster than NDVI

Pixel B (never burned):
  - smooth seasonal NDVI, no sudden drops
  - Z-score stays near 0 throughout 2017-2025
  - no correlation with fire labels

Pixel C (repeat burns):
  - multiple NDVI drops across years
  - Z-score does not fully return to 0 between events
  - later fires start from a lower baseline

If any of these fail -> stop here, do not proceed to sampling.
""")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 38.0 MB/s eta 0:00:00
Pixel A (single burn 2023):   34.56704, 31.19135
Pixel B (never burned):       34.39582, 31.32565


KeyboardInterrupt: 

General statistics to get to know my data better (the data on gee)

## Changed Pixel Resolution to 10m

I have updated the `SCALE` variable in all relevant code cells from 20m to 10m. This change will ensure that all subsequent image processing, exports, and analyses are performed at a 10-meter pixel resolution, providing finer detail in the results.